# Lab 2: RAG 파이프라인 구축

RAG(Retrieval-Augmented Generation)의 두 번째 단계입니다.  
ChromaDB에서 검색한 문서를 LLM에 컨텍스트로 전달하여 **근거 있는 답변**을 생성합니다.

## 학습 목표

1. RAG 전체 파이프라인 이해: **Retriever → Augment → Generate**
2. `ManufacturingRAG` 클래스 구현 (API 없이도 동작)
3. 제조 AI 질문 5개 이상으로 RAG 파이프라인 테스트
4. **RAG vs 직접 답변 비교** (환각 방지 효과 확인)
5. 간단한 품질 평가 및 **Streamlit 연결 개념** 이해

## RAG 파이프라인 전체 흐름

```
[검색 단계 - Retrieval]
사용자 질문 → 임베딩 → ChromaDB 유사도 검색 → 관련 문서 Top-K
                                                        ↓
[증강 단계 - Augmentation]                       프롬프트 조합
                                    (검색된 문서 + 질문 → 프롬프트 템플릿)
                                                        ↓
[생성 단계 - Generation]                     LLM (GPT/Claude) → 최종 답변
```

> **왜 RAG인가?**  
> 일반 LLM: 학습 데이터 기반 → 환각(Hallucination) 위험  
> RAG: 실시간 문서 검색 기반 → 근거 있는 정확한 답변

## 1. 환경 설정 및 ChromaDB 인덱스 준비

Lab 1에서 만든 ChromaDB 인덱스를 재사용합니다.

In [1]:
# 필요한 패키지 설치 (최초 1회만 실행)
# !pip install chromadb sentence-transformers pandas

In [2]:
# 라이브러리 임포트
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import json
import os
from typing import List, Dict, Optional

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


In [3]:
# ChromaDB 인덱스 준비
# Lab 1에서 영구 저장한 경우 PersistentClient 사용
# Lab 1을 건너뛴 경우 이 셀에서 직접 인덱스를 생성합니다

COLLECTION_NAME = "manufacturing_docs_v1"
PERSIST_PATH = "./chroma_db_manufacturing"

# 임베딩 모델 로드
print("임베딩 모델 로드 중...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 영구 저장된 인덱스가 있는지 확인
if os.path.exists(PERSIST_PATH):
    print(f"✅ 저장된 인덱스 발견: {PERSIST_PATH}")
    client = chromadb.PersistentClient(path=PERSIST_PATH)
    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"   컬렉션 로드 완료: {collection.count()}개 문서")
    except Exception as e:
        print(f"컬렉션 없음. 새로 생성합니다: {e}")
        collection = None
else:
    print("저장된 인덱스 없음. 메모리 클라이언트 사용")
    client = chromadb.EphemeralClient()
    collection = None

# 컬렉션이 없으면 새로 생성
if collection is None:
    print("인덱스를 새로 생성합니다...")
    
    # Lab 1과 동일한 샘플 문서
    manufacturing_docs = [
        {"id": "doc01", "text": "CNC 밀링머신 진동 이상: 스핀들 베어링 마모 시 100~300Hz 대역에서 비정상 피크 발생. 진동값 3mm/s 초과 시 가동 중단 및 점검 필요.", "source": "설비매뉴얼", "topic": "진동이상"},
        {"id": "doc02", "text": "용접 품질 검사: 기공 결함은 X-ray에서 원형 음영, KS B 0845 기준 직경 3mm 이상 불합격. 초음파 검사 월 1회 실시.", "source": "품질기준서", "topic": "용접품질"},
        {"id": "doc03", "text": "AutoEncoder 이상탐지: 정상 데이터만 학습, 재구성 오차가 임계값(표준편차 3배) 초과 시 이상 판정. 민감도 조정 가능.", "source": "AI기술가이드", "topic": "이상탐지"},
        {"id": "doc04", "text": "스마트공장 ROI(KAMP): 300개사 평균 생산성 33.6% 향상, 불량률 44.4% 감소. 초기 비용 1~3억, 평균 회수 2.8년.", "source": "KAMP사례집", "topic": "스마트공장ROI"},
        {"id": "doc05", "text": "프레스 금형 마모: 타발 50만회 도달 시 점검. 버 높이 0.3mm 초과 또는 치수 편차 ±0.1mm 초과 시 금형 교체.", "source": "작업표준서", "topic": "금형관리"},
        {"id": "doc06", "text": "컨베이어 벨트 장력: 처짐량 벨트 길이의 1~3% 유지. 부족 시 슬립, 과도 시 베어링 수명 단축. 분기별 측정.", "source": "설비매뉴얼", "topic": "컨베이어관리"},
        {"id": "doc07", "text": "YOLOv8 표면 결함 검사: 스크래치/크랙/이물질 실시간 탐지. 신뢰도 0.85 이상만 불량 판정, 오탐 최소화.", "source": "AI기술가이드", "topic": "비전검사"},
        {"id": "doc08", "text": "윤활유 교체 주기: 운전 2000시간 또는 6개월 중 먼저 도달 시 교체. 점도지수(VI) 80 미만 저하 시 즉시 교체.", "source": "설비매뉴얼", "topic": "윤활관리"},
        {"id": "doc09", "text": "예지보전(PdM): 1단계 IoT 데이터 수집, 2단계 이상탐지, 3단계 RUL 수명예측, 4단계 자동 스케줄링. 단계별 ROI 측정 필수.", "source": "AI기술가이드", "topic": "예지보전"},
        {"id": "doc10", "text": "사출성형기 온도: 배럴 온도 설정값 대비 ±5°C 유지. 편차 10°C 이상 시 히터/쿨링 점검. 1분 단위 로그 기록.", "source": "설비매뉴얼", "topic": "온도관리"},
        {"id": "doc11", "text": "전기 모터 관리: IE3 효율 등급 이상 사용 권장. 전류 불균형 5% 초과 시 권선 절연 열화 의심, 절연저항 1MΩ 이상 유지.", "source": "전기설비기준", "topic": "모터관리"},
        {"id": "doc12", "text": "FFT 베어링 진단: BPFO(외륜결함)=0.4×회전수×볼수, BPFI(내륜결함)=0.6×회전수×볼수. 해당 주파수 피크 시 결함 판정.", "source": "진단기술서", "topic": "FFT진단"},
    ]
    
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    
    collection = client.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
    texts = [d["text"] for d in manufacturing_docs]
    embeddings = model.encode(texts).tolist()
    collection.add(
        ids=[d["id"] for d in manufacturing_docs],
        embeddings=embeddings,
        documents=texts,
        metadatas=[{"source": d["source"], "topic": d["topic"]} for d in manufacturing_docs]
    )
    print(f"✅ 새 인덱스 생성 완료: {collection.count()}개 문서")

임베딩 모델 로드 중...


✅ 저장된 인덱스 발견: ./chroma_db_manufacturing
   컬렉션 로드 완료: 12개 문서


## 2. RAG 컴포넌트 이해

RAG는 3개의 핵심 컴포넌트로 구성됩니다:

| 컴포넌트 | 역할 | 이 실습에서 |
|---------|------|-------------|
| **Retriever** | ChromaDB에서 관련 문서 검색 | `collection.query()` |
| **Augmenter** | 검색 문서 + 질문 → 프롬프트 조합 | 프롬프트 템플릿 함수 |
| **Generator** | LLM으로 최종 답변 생성 | Mock LLM (API 없이 학습 가능) |

실제 서비스에서는 Generator에 `openai.ChatCompletion` 또는 `anthropic.messages`를 사용합니다.

## 3. ManufacturingRAG 클래스 구현

In [4]:
class ManufacturingRAG:
    """
    제조 도메인 특화 RAG 파이프라인.
    
    LLM API 없이도 프롬프트 조합 방식으로 동작합니다.
    실제 서비스에서는 generate() 메서드에 OpenAI/Claude API를 연결하세요.
    """
    
    def __init__(self, collection, embed_model, top_k: int = 3):
        """
        Args:
            collection: ChromaDB 컬렉션 객체
            embed_model: SentenceTransformer 임베딩 모델
            top_k: 검색할 관련 문서 수 (기본 3개)
        """
        self.collection = collection
        self.embed_model = embed_model
        self.top_k = top_k
    
    def retrieve(self, query: str, n_results: Optional[int] = None) -> List[Dict]:
        """
        [Retrieval 단계] ChromaDB에서 관련 문서를 검색합니다.
        
        Args:
            query: 사용자 질문
            n_results: 검색할 문서 수 (None이면 self.top_k 사용)
        
        Returns:
            검색된 문서 목록: [{'text': ..., 'source': ..., 'topic': ..., 'similarity': ...}]
        """
        k = n_results or self.top_k
        
        # 쿼리를 임베딩으로 변환
        query_embedding = self.embed_model.encode([query]).tolist()
        
        # ChromaDB에서 유사도 검색
        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )
        
        # 검색 결과를 표준 형식으로 변환
        retrieved_docs = []
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ):
            retrieved_docs.append({
                "text": doc,
                "source": meta.get("source", "알 수 없음"),
                "topic": meta.get("topic", ""),
                "similarity": round(1 - dist, 4)  # cosine 거리 → 유사도
            })
        
        return retrieved_docs
    
    def augment(self, query: str, context_docs: List[Dict]) -> str:
        """
        [Augmentation 단계] 검색된 문서와 질문을 조합하여 프롬프트를 생성합니다.
        
        Args:
            query: 사용자 질문
            context_docs: retrieve()에서 반환된 문서 목록
        
        Returns:
            LLM에 전달할 완성된 프롬프트 문자열
        """
        # 검색된 문서를 번호 붙여 나열
        context_text = ""
        for i, doc in enumerate(context_docs, 1):
            context_text += f"[문서 {i}] 출처: {doc['source']} (유사도: {doc['similarity']})\n"
            context_text += f"{doc['text']}\n\n"
        
        # 제조 AI 특화 프롬프트 템플릿
        prompt = f"""당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요.

=== 참고 문서 ===
{context_text.strip()}

=== 질문 ===
{query}

=== 답변 ==="""
        
        return prompt
    
    def generate(self, prompt: str, use_mock: bool = True) -> str:
        """
        [Generation 단계] LLM으로 답변을 생성합니다.
        
        Args:
            prompt: augment()에서 생성된 프롬프트
            use_mock: True이면 Mock 응답 반환 (API 없이 학습 가능)
                      False이면 실제 OpenAI API 호출
        
        Returns:
            LLM 생성 답변 문자열
        """
        if use_mock:
            # Mock LLM: 프롬프트에서 참고 문서 내용을 요약하여 반환
            # 실제 서비스에서는 이 부분을 OpenAI API로 교체합니다
            lines = prompt.split("\n")
            doc_lines = [l for l in lines if l.startswith("[문서") or (len(l) > 20 and "===" not in l and "출처:" not in l)]
            
            mock_answer = "[Mock 답변] 참고 문서 기반 응답:\n"
            for line in doc_lines[:3]:  # 처음 3줄만 요약
                if len(line) > 20:
                    mock_answer += f"  • {line[:100]}\n"
            mock_answer += "\n※ 실제 서비스에서는 GPT-4.1-mini/Claude 등 LLM이 자연스러운 답변을 생성합니다."
            return mock_answer
        else:
            # 실제 OpenAI API 연동 예시
            # from openai import OpenAI
            # client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            # response = client.chat.completions.create(
            #     model="gpt-4.1-mini",
            #     messages=[{"role": "user", "content": prompt}],
            #     temperature=0  # 일관된 답변을 위해 temperature=0
            # )
            # return response.choices[0].message.content
            raise NotImplementedError("OPENAI_API_KEY 설정 후 주석 해제하여 사용하세요.")
    
    def query(self, question: str, verbose: bool = False) -> Dict:
        """
        RAG 파이프라인 전체 실행: 검색 → 증강 → 생성
        
        Args:
            question: 사용자 질문
            verbose: True이면 각 단계 과정 출력
        
        Returns:
            {'question', 'retrieved_docs', 'prompt', 'answer'} 딕셔너리
        """
        # 1단계: Retrieval (관련 문서 검색)
        if verbose:
            print(f"[1/3] 검색 중... '{question}'")
        docs = self.retrieve(question)
        
        # 2단계: Augmentation (프롬프트 조합)
        if verbose:
            print(f"[2/3] 프롬프트 생성 중... (검색된 문서 {len(docs)}개)")
        prompt = self.augment(question, docs)
        
        # 3단계: Generation (LLM 답변 생성)
        if verbose:
            print(f"[3/3] 답변 생성 중...")
        answer = self.generate(prompt)
        
        return {
            "question": question,
            "retrieved_docs": docs,
            "prompt": prompt,
            "answer": answer
        }


# RAG 인스턴스 생성
rag = ManufacturingRAG(
    collection=collection,
    embed_model=model,
    top_k=3  # 상위 3개 문서 검색
)

print("✅ ManufacturingRAG 인스턴스 생성 완료")
print(f"   검색 문서 수: Top-{rag.top_k}")
print(f"   인덱스 문서 총 수: {collection.count()}개")

✅ ManufacturingRAG 인스턴스 생성 완료
   검색 문서 수: Top-3
   인덱스 문서 총 수: 12개


## 4. RAG 파이프라인 단계별 실습

In [5]:
# 단계별로 RAG 파이프라인 실행
test_question = "CNC 머신에서 베어링 진동 이상이 발생하면 어떤 기준으로 판단하나요?"

print("=" * 60)
print("RAG 파이프라인 단계별 실행")
print("=" * 60)

# --- 1단계: Retrieval ---
print("\n[1단계] RETRIEVAL - 관련 문서 검색")
print("-" * 40)
retrieved = rag.retrieve(test_question)
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] 유사도: {doc['similarity']} | 주제: {doc['topic']} | 출처: {doc['source']}")
    print(f"       {doc['text'][:80]}...")

# --- 2단계: Augmentation ---
print("\n[2단계] AUGMENTATION - 프롬프트 조합")
print("-" * 40)
prompt = rag.augment(test_question, retrieved)
print("생성된 프롬프트 (앞 400자):")
print(prompt[:400] + "...")

# --- 3단계: Generation ---
print("\n[3단계] GENERATION - 답변 생성")
print("-" * 40)
answer = rag.generate(prompt)
print(answer)

RAG 파이프라인 단계별 실행

[1단계] RETRIEVAL - 관련 문서 검색
----------------------------------------


  [1] 유사도: 0.6873 | 주제: 진동이상 | 출처: 설비매뉴얼
       CNC 밀링머신 진동 이상 징후: 스핀들 베어링 마모 시 100~300Hz 대역에서 비정상적인 피크가 발생합니다. 진동값이 3mm/s를 초과하면...
  [2] 유사도: 0.4794 | 주제: 금형관리 | 출처: 작업표준서
       프레스 금형 마모 점검 기준: 타발 횟수 50만회 도달 시 금형 마모 점검을 실시합니다. 버(Burr) 높이가 0.3mm 초과하거나 치수 편차가...
  [3] 유사도: 0.4202 | 주제: 예지보전 | 출처: AI기술가이드
       예지보전(PdM) 구현 단계: 1단계 데이터 수집(IoT 센서), 2단계 이상탐지 모델(AutoEncoder/FFT), 3단계 수명예측(RUL ...

[2단계] AUGMENTATION - 프롬프트 조합
----------------------------------------
생성된 프롬프트 (앞 400자):
당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요.

=== 참고 문서 ===
[문서 1] 출처: 설비매뉴얼 (유사도: 0.6873)
CNC 밀링머신 진동 이상 징후: 스핀들 베어링 마모 시 100~300Hz 대역에서 비정상적인 피크가 발생합니다. 진동값이 3mm/s를 초과하면 즉시 가동을 중단하고 점검이 필요합니다. SKF-6205 베어링 기준.

[문서 2] 출처: 작업표준서 (유사도: 0.4794)
프레스 금형 마모 점검 기준: 타발 횟수 50만회 도달 시 금형 마모 점검을 실시합니다. 버(Burr) 높이가 0.3mm 초과하거나 치수 편차가 ±0.1mm를 넘으면 금형 교체가 필요합니다.

[문...

[3단계] GENERATION - 답변 생성
----------------------------------------
[Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어

## 5. 제조 AI 질문 5개 이상으로 RAG 테스트

In [6]:
# 제조 현장 실제 질문 6개로 RAG 파이프라인 테스트
test_questions = [
    "스마트공장 도입하면 생산성이 얼마나 향상되나요?",
    "AutoEncoder로 설비 이상탐지를 어떻게 구현하나요?",
    "용접 불량 검사 기준과 주기는 어떻게 되나요?",
    "예지보전 시스템 구축 단계를 설명해주세요",
    "FFT 분석으로 베어링 결함을 어떻게 진단하나요?",
    "윤활유 교체 시기는 어떻게 결정하나요?",
]

print("=" * 65)
print("제조 AI 질문 RAG 테스트 결과")
print("=" * 65)

results_for_comparison = []  # RAG vs 직접 답변 비교를 위해 저장

for i, question in enumerate(test_questions, 1):
    result = rag.query(question)
    
    print(f"\n[Q{i}] {question}")
    print(f"  검색된 문서: {[doc['topic'] for doc in result['retrieved_docs']]}")
    print(f"  최고 유사도: {result['retrieved_docs'][0]['similarity']}")
    print(f"  답변 미리보기: {result['answer'][:120]}...")
    print("-" * 65)
    
    results_for_comparison.append(result)

print("\n✅ 6개 질문 테스트 완료")

제조 AI 질문 RAG 테스트 결과



[Q1] 스마트공장 도입하면 생산성이 얼마나 향상되나요?
  검색된 문서: ['스마트공장ROI', '모터관리', '윤활관리']
  최고 유사도: 0.7189
  답변 미리보기: [Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
  • 문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요...
-----------------------------------------------------------------

[Q2] AutoEncoder로 설비 이상탐지를 어떻게 구현하나요?
  검색된 문서: ['이상탐지', '예지보전', '비전검사']
  최고 유사도: 0.697
  답변 미리보기: [Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
  • 문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요...
-----------------------------------------------------------------

[Q3] 용접 불량 검사 기준과 주기는 어떻게 되나요?
  검색된 문서: ['용접품질', '금형관리', '비전검사']
  최고 유사도: 0.6014
  답변 미리보기: [Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
  • 문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요...
-----------------------------------------------------------------

[Q4] 예지보전 시스템 구축 단계를 설명해주세요
  검색된 문서: ['예지보전', '금형관리', '이상탐지']
  최고 유사도: 0.4965
  답변 미리보기: [Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시


[Q5] FFT 분석으로 베어링 결함을 어떻게 진단하나요?
  검색된 문서: ['FFT진단', '용접품질', '비전검사']
  최고 유사도: 0.649
  답변 미리보기: [Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
  • 문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요...
-----------------------------------------------------------------

[Q6] 윤활유 교체 시기는 어떻게 결정하나요?
  검색된 문서: ['윤활관리', '온도관리', '진동이상']
  최고 유사도: 0.2494
  답변 미리보기: [Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
  • 문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요...
-----------------------------------------------------------------

✅ 6개 질문 테스트 완료


## 6. RAG vs 직접 답변 비교

RAG의 핵심 가치: **문서 기반 근거 있는 답변 vs 학습 데이터 기반 답변(환각 위험)**

In [7]:
def direct_answer(question: str) -> str:
    """
    RAG 없이 직접 답변하는 Mock LLM 시뮬레이션.
    실제로는 문서 없이 LLM만 호출하는 경우를 시뮬레이션합니다.
    """
    # 일반 LLM의 한계 시뮬레이션: 구체적 수치는 임의로 생성 (환각)
    mock_responses = {
        "스마트공장": "스마트공장 도입 시 생산성이 향상됩니다. (구체적 수치는 기업마다 다릅니다)",
        "AutoEncoder": "AutoEncoder는 비지도 학습 방법으로 이상탐지에 사용됩니다.",
        "용접": "용접 품질 검사는 정기적으로 실시해야 합니다.",
        "예지보전": "예지보전은 데이터 기반으로 고장을 예측하는 기술입니다.",
        "FFT": "FFT 분석은 주파수 영역에서 신호를 분석하는 방법입니다.",
        "윤활유": "윤활유는 정기적으로 교체해야 합니다."
    }
    
    for keyword, response in mock_responses.items():
        if keyword in question:
            return f"[직접 답변 - 일반 LLM] {response}\n※ 구체적 수치나 기준 없음 (환각 위험)"
    return "[직접 답변] 관련 정보를 찾을 수 없습니다."


# RAG vs 직접 답변 비교 테이블 생성
comparison_question = "스마트공장 도입하면 생산성이 얼마나 향상되나요?"

rag_result = rag.query(comparison_question)
direct_result = direct_answer(comparison_question)

print(f"질문: {comparison_question}")
print("=" * 65)
print("\n[직접 답변 (RAG 없이)]")
print(direct_result)
print("\n[RAG 답변 (문서 기반)]")
print(f"검색된 근거 문서: {rag_result['retrieved_docs'][0]['text'][:100]}...")
print(rag_result['answer'])

질문: 스마트공장 도입하면 생산성이 얼마나 향상되나요?

[직접 답변 (RAG 없이)]
[직접 답변 - 일반 LLM] 스마트공장 도입 시 생산성이 향상됩니다. (구체적 수치는 기업마다 다릅니다)
※ 구체적 수치나 기준 없음 (환각 위험)

[RAG 답변 (문서 기반)]
검색된 근거 문서: 스마트공장 도입 ROI 사례(KAMP): 중소 제조업체 300개사 평균 생산성 33.6% 향상, 품질 불량률 44.4% 감소. 초기 구축 비용 1~3억원, 평균 투자 회수 기간 2...
[Mock 답변] 참고 문서 기반 응답:
  • 당신은 제조 현장 AI 어시스턴트입니다. 아래 참고 문서를 기반으로 질문에 답하세요.
  • 문서 내용에 없는 정보는 '참고 문서에 해당 정보가 없습니다'라고 답하세요.
  • [문서 1] 출처: KAMP사례집 (유사도: 0.7189)

※ 실제 서비스에서는 GPT-4.1-mini/Claude 등 LLM이 자연스러운 답변을 생성합니다.


In [8]:
# RAG vs 직접 답변 특성 비교 테이블
comparison_data = {
    "항목": ["답변 근거", "환각(Hallucination)", "최신 정보 반영", "수치 정확도", "출처 표시", "도메인 특화"],
    "일반 LLM (RAG 없음)": [
        "학습 데이터 (과거)",
        "⚠️ 높음",
        "❌ 어려움",
        "⚠️ 불확실",
        "❌ 없음",
        "△ 범용적"
    ],
    "RAG 시스템": [
        "검색된 실시간 문서",
        "✅ 낮음 (문서 기반)",
        "✅ 문서 추가로 즉시 반영",
        "✅ 문서의 수치 그대로",
        "✅ 출처 명시 가능",
        "✅ 도메인 문서 특화"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n=== RAG vs 일반 LLM 비교 ===\n")
print(df_comparison.to_string(index=False))


=== RAG vs 일반 LLM 비교 ===

               항목 일반 LLM (RAG 없음)        RAG 시스템
            답변 근거     학습 데이터 (과거)     검색된 실시간 문서
환각(Hallucination)           ⚠️ 높음   ✅ 낮음 (문서 기반)
         최신 정보 반영           ❌ 어려움 ✅ 문서 추가로 즉시 반영
           수치 정확도          ⚠️ 불확실   ✅ 문서의 수치 그대로
            출처 표시            ❌ 없음     ✅ 출처 명시 가능
           도메인 특화           △ 범용적    ✅ 도메인 문서 특화


## 7. 검색 품질 간단 평가

In [9]:
# 검색 품질 수동 평가
# 각 질문에 대한 예상 정답 주제를 설정하여 Precision 계산

evaluation_set = [
    {"question": "CNC 베어링 진동 기준", "expected_topics": {"진동이상", "FFT진단"}},
    {"question": "스마트공장 투자 ROI", "expected_topics": {"스마트공장ROI"}},
    {"question": "AutoEncoder 이상탐지", "expected_topics": {"이상탐지"}},
    {"question": "용접 검사 기준", "expected_topics": {"용접품질"}},
    {"question": "예지보전 단계", "expected_topics": {"예지보전"}},
]

print("=" * 60)
print("RAG 검색 품질 평가 (Precision@3)")
print("=" * 60)

eval_results = []
for eval_item in evaluation_set:
    retrieved = rag.retrieve(eval_item["question"], n_results=3)
    retrieved_topics = {doc["topic"] for doc in retrieved}
    
    # 예상 주제가 검색 결과에 포함됐는지 확인
    hit = bool(retrieved_topics & eval_item["expected_topics"])
    top_similarity = retrieved[0]["similarity"] if retrieved else 0
    
    eval_results.append({
        "질문": eval_item["question"],
        "예상 주제": str(eval_item["expected_topics"]),
        "검색된 주제": str(retrieved_topics),
        "적중": "✅" if hit else "❌",
        "최고 유사도": f"{top_similarity:.3f}"
    })

eval_df = pd.DataFrame(eval_results)
print(eval_df.to_string(index=False))

# 전체 Precision 계산
total_hits = sum(1 for r in eval_results if r["적중"] == "✅")
precision = total_hits / len(eval_results)
print(f"\n전체 Precision@3: {total_hits}/{len(eval_results)} = {precision:.1%}")

RAG 검색 품질 평가 (Precision@3)


              질문             예상 주제                       검색된 주제 적중 최고 유사도
   CNC 베어링 진동 기준 {'FFT진단', '진동이상'}   {'컨베이어관리', '금형관리', '진동이상'}  ✅  0.633
    스마트공장 투자 ROI      {'스마트공장ROI'} {'예지보전', '스마트공장ROI', '모터관리'}  ✅  0.618
AutoEncoder 이상탐지          {'이상탐지'}     {'비전검사', '예지보전', '이상탐지'}  ✅  0.718
        용접 검사 기준          {'용접품질'}     {'비전검사', '용접품질', '금형관리'}  ✅  0.577
         예지보전 단계          {'예지보전'}     {'예지보전', '온도관리', '금형관리'}  ✅  0.352

전체 Precision@3: 5/5 = 100.0%


## 8. Streamlit 신호등 대시보드 연결 개념

이 RAG 파이프라인을 Streamlit 챗봇으로 확장하는 방법을 설명합니다.

In [10]:
# Streamlit 앱 구조 미리보기 (코드 설명용)
streamlit_code_preview = '''
# src/streamlit_app.py (실행: streamlit run src/streamlit_app.py)

import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer

# 앱 제목
st.title("🏭 제조 AI 매뉴얼 챗봇")
st.caption("설비 매뉴얼 기반 RAG Q&A 시스템 | KOREATECH")

# RAG 초기화 (st.cache_resource로 한 번만 실행)
@st.cache_resource
def load_rag():
    client = chromadb.PersistentClient(path="./chroma_db_manufacturing")
    collection = client.get_collection("manufacturing_docs_v1")
    model = SentenceTransformer(\'paraphrase-multilingual-MiniLM-L12-v2\')
    return ManufacturingRAG(collection, model, top_k=3)

rag = load_rag()

# 대화 이력 초기화
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 대화 표시
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# 사용자 입력 처리
if prompt := st.chat_input("설비 매뉴얼에 질문하세요..."):
    # 사용자 메시지 표시
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})
    
    # RAG로 답변 생성
    result = rag.query(prompt)
    
    # AI 답변 표시
    with st.chat_message("assistant"):
        st.markdown(result["answer"])
        
        # 출처 표시 (신뢰도 포함)
        with st.expander("📄 참조 문서 및 신뢰도"):
            for doc in result["retrieved_docs"]:
                similarity = doc[\'similarity\']
                # 신호등 색상: 높음=녹색, 중간=노랑, 낮음=빨강
                color = "🟢" if similarity > 0.8 else ("🟡" if similarity > 0.6 else "🔴")
                st.write(f"{color} **{doc[\'source\']}** (유사도: {similarity:.3f})")
                st.write(f"  {doc[\'text\'][:100]}...")
    
    st.session_state.messages.append({"role": "assistant", "content": result["answer"]})
'''

print("=== Streamlit 챗봇 코드 구조 (설명용) ===")
print(streamlit_code_preview)
print("\n실제 파일 위치: src/streamlit_app.py")
print("실행 명령어: streamlit run src/streamlit_app.py")

=== Streamlit 챗봇 코드 구조 (설명용) ===

# src/streamlit_app.py (실행: streamlit run src/streamlit_app.py)

import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer

# 앱 제목
st.title("🏭 제조 AI 매뉴얼 챗봇")
st.caption("설비 매뉴얼 기반 RAG Q&A 시스템 | KOREATECH")

# RAG 초기화 (st.cache_resource로 한 번만 실행)
@st.cache_resource
def load_rag():
    client = chromadb.PersistentClient(path="./chroma_db_manufacturing")
    collection = client.get_collection("manufacturing_docs_v1")
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return ManufacturingRAG(collection, model, top_k=3)

rag = load_rag()

# 대화 이력 초기화
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 대화 표시
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# 사용자 입력 처리
if prompt := st.chat_input("설비 매뉴얼에 질문하세요..."):
    # 사용자 메시지 표시
    with st.chat_message("user"):
        st.markdown(prompt)
  

In [11]:
# 신호등 대시보드 개념 시뮬레이션
# 유사도 점수에 따라 신뢰도를 신호등으로 표시

def get_traffic_light(similarity: float) -> str:
    """유사도 점수 → 신호등 색상 변환"""
    if similarity >= 0.80:
        return "🟢 높음 (신뢰 가능)"
    elif similarity >= 0.60:
        return "🟡 중간 (주의 필요)"
    else:
        return "🔴 낮음 (추가 확인 필요)"

# 테스트 쿼리들의 신뢰도 시각화
demo_queries = [
    "CNC 베어링 진동 임계값",
    "사출기 온도 관리",
    "인공지능 스마트공장 ROI",
    "프레스 금형 수명",
    "용접 불량 KS 기준",
]

print("=" * 60)
print("신호등 신뢰도 대시보드 시뮬레이션")
print("=" * 60)

for q in demo_queries:
    docs = rag.retrieve(q, n_results=1)
    sim = docs[0]['similarity']
    traffic = get_traffic_light(sim)
    print(f"  쿼리: '{q}'")
    print(f"  신뢰도: {traffic} ({sim:.3f}) | 출처: {docs[0]['source']}")
    print()

신호등 신뢰도 대시보드 시뮬레이션
  쿼리: 'CNC 베어링 진동 임계값'
  신뢰도: 🟡 중간 (주의 필요) (0.624) | 출처: 설비매뉴얼

  쿼리: '사출기 온도 관리'
  신뢰도: 🟡 중간 (주의 필요) (0.774) | 출처: 설비매뉴얼

  쿼리: '인공지능 스마트공장 ROI'
  신뢰도: 🔴 낮음 (추가 확인 필요) (0.518) | 출처: AI기술가이드

  쿼리: '프레스 금형 수명'
  신뢰도: 🔴 낮음 (추가 확인 필요) (0.314) | 출처: 작업표준서

  쿼리: '용접 불량 KS 기준'
  신뢰도: 🔴 낮음 (추가 확인 필요) (0.450) | 출처: 품질기준서



## 9. 핵심 정리

### RAG 파이프라인 완성

```
사용자 질문
    ↓
[Retrieval]  질문 → 임베딩 → ChromaDB 유사도 검색 → 관련 문서 Top-K
    ↓
[Augmentation]  검색 문서 + 질문 → 프롬프트 템플릿 조합
    ↓
[Generation]  프롬프트 → LLM (GPT-4.1-mini / Claude / 로컬 모델) → 답변
    ↓
최종 답변 (출처 표시 + 신뢰도 신호등)
```

### 실제 서비스 배포 시 변경사항

| 항목 | 이 Lab (학습용) | 실서비스 |
|------|----------------|----------|
| ChromaDB | `EphemeralClient()` | `PersistentClient(path=...)` |
| LLM | Mock 응답 | OpenAI/Claude API |
| 문서 | 12개 샘플 | 실제 설비 매뉴얼 PDF |
| 임베딩 | 다국어 MiniLM | `text-embedding-3-small` (더 정확) |
| 검색 방식 | Dense만 | BM25 + Dense 하이브리드 |

### 다음 단계
- `src/streamlit_app.py`를 실행하여 Streamlit 챗봇 완성
- `X1-S4: ReAct Agent`에서 이 RAG를 에이전트의 Tool로 등록